# Обработка данных Video Game Sales с использованием PySpark

## Цель работы

В рамках данной контрольной точки необходимо выполнить обработку датасета **Video Game Sales** с использованием PySpark.

Основные задачи:

1. Прочитать исходный датасет.
2. Выполнить нормализацию данных.
3. Рассчитать общие продажи игр по годам.
4. Рассчитать продажи по годам, платформам и регионам.
5. Рассчитать доли продаж по годам, платформам и регионам.
6. Рассчитать долю продаж по издателю.
7. Создать столбец с суммой продаж каждой игры.
8. Сохранить итоговый датасет в формате Parquet.

В работе используется Apache Spark, запущенный в Docker-кластере.

## 1. Подключение к Spark-кластеру

На первом этапе создадим `SparkSession`.  
Ноутбук Jupyter будет выступать в роли клиента, а вычисления будут выполняться в Spark-кластере.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    round as spark_round,
    when
)
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.builder \
    .appName("VideoGameSalesProcessing") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

spark

## 2. Чтение исходного датасета

На данном этапе прочитаем CSV-файл `vgsales.csv`, который находится в папке `data/raw`.

При чтении указываем:
- `header=True`, чтобы первая строка использовалась как названия столбцов;
- `inferSchema=True`, чтобы Spark автоматически определил типы данных.

In [4]:
csv_path = "/home/jovyan/work/data/raw/vgsales.csv"

df = spark.read.csv(
    csv_path,
    header=True,
    inferSchema=True
)

df.show(5)

+----+--------------------+--------+----+------------+---------+--------+--------+--------+-----------+------------+
|Rank|                Name|Platform|Year|       Genre|Publisher|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|
+----+--------------------+--------+----+------------+---------+--------+--------+--------+-----------+------------+
|   1|          Wii Sports|     Wii|2006|      Sports| Nintendo|   41.49|   29.02|    3.77|       8.46|       82.74|
|   2|   Super Mario Bros.|     NES|1985|    Platform| Nintendo|   29.08|    3.58|    6.81|       0.77|       40.24|
|   3|      Mario Kart Wii|     Wii|2008|      Racing| Nintendo|   15.85|   12.88|    3.79|       3.31|       35.82|
|   4|   Wii Sports Resort|     Wii|2009|      Sports| Nintendo|   15.75|   11.01|    3.28|       2.96|        33.0|
|   5|Pokemon Red/Pokem...|      GB|1996|Role-Playing| Nintendo|   11.27|    8.89|   10.22|        1.0|       31.37|
+----+--------------------+--------+----+------------+---------+

In [5]:
df.printSchema()

root
 |-- Rank: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Platform: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- NA_Sales: double (nullable = true)
 |-- EU_Sales: double (nullable = true)
 |-- JP_Sales: double (nullable = true)
 |-- Other_Sales: double (nullable = true)
 |-- Global_Sales: double (nullable = true)



## 3. Нормализация данных

Перед расчётами необходимо подготовить датасет:

- удалить строки с пропущенными значениями в ключевых полях;
- привести `Year` к целочисленному типу;
- привести столбцы продаж к числовому типу;
- убрать строки с некорректным годом;
- проверить результат очистки.

Это нужно, чтобы дальнейшие группировки и расчёты выполнялись корректно.

In [27]:
from pyspark.sql.functions import col

In [28]:
df_clean = df.select(
    col("Rank").cast("int").alias("Rank"),
    col("Name").alias("Name"),
    col("Platform").alias("Platform"),
    col("Year").cast("int").alias("Year"),
    col("Genre").alias("Genre"),
    col("Publisher").alias("Publisher"),
    col("NA_Sales").cast("double").alias("NA_Sales"),
    col("EU_Sales").cast("double").alias("EU_Sales"),
    col("JP_Sales").cast("double").alias("JP_Sales"),
    col("Other_Sales").cast("double").alias("Other_Sales"),
    col("Global_Sales").cast("double").alias("Global_Sales")
)

In [29]:
df_clean = df_clean.dropna(
    subset=[
        "Name",
        "Platform",
        "Year",
        "Genre",
        "Publisher",
        "NA_Sales",
        "EU_Sales",
        "JP_Sales",
        "Other_Sales",
        "Global_Sales"
    ]
)

In [30]:
df_clean = df_clean.filter(col("Year") > 0)

In [31]:
print("Количество строк до очистки:", df.count())
print("Количество строк после очистки:", df_clean.count())

Количество строк до очистки: 16598
Количество строк после очистки: 16327


In [32]:
df_clean.printSchema()

root
 |-- Rank: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Platform: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- NA_Sales: double (nullable = true)
 |-- EU_Sales: double (nullable = true)
 |-- JP_Sales: double (nullable = true)
 |-- Other_Sales: double (nullable = true)
 |-- Global_Sales: double (nullable = true)



## 4. Создание столбца с общими продажами игр по годам

Теперь рассчитаем суммарные глобальные продажи игр для каждого года.

Для этого:
- сгруппируем данные по столбцу `Year`;
- посчитаем сумму `Global_Sales`;
- присоединим полученный результат обратно к основному датасету.

В результате у каждой строки появится новый столбец `Total_Sales_By_Year`.

In [34]:
from pyspark.sql.functions import sum as spark_sum, round as spark_round

In [35]:
sales_by_year = df_clean.groupBy("Year") \
    .agg(
        spark_sum("Global_Sales").alias("Total_Sales_By_Year")
    )

sales_by_year = sales_by_year.withColumn(
    "Total_Sales_By_Year",
    spark_round(col("Total_Sales_By_Year"), 2)
)

sales_by_year.orderBy("Year").show(10)

+----+-------------------+
|Year|Total_Sales_By_Year|
+----+-------------------+
|1980|              11.38|
|1981|              35.77|
|1982|              28.86|
|1983|              16.79|
|1984|              50.36|
|1985|              53.94|
|1986|              37.07|
|1987|              21.74|
|1988|              47.22|
|1989|              73.45|
+----+-------------------+
only showing top 10 rows



In [36]:
df_step_3 = df_clean.join(
    sales_by_year,
    on="Year",
    how="left"
)

df_step_3.select(
    "Name",
    "Platform",
    "Year",
    "Global_Sales",
    "Total_Sales_By_Year"
).orderBy("Year").show(10)

+---------------+--------+----+------------+-------------------+
|           Name|Platform|Year|Global_Sales|Total_Sales_By_Year|
+---------------+--------+----+------------+-------------------+
|Missile Command|    2600|1980|        2.76|              11.38|
|      Asteroids|    2600|1980|        4.31|              11.38|
|        Kaboom!|    2600|1980|        1.15|              11.38|
|       Defender|    2600|1980|        1.05|              11.38|
|         Boxing|    2600|1980|        0.77|              11.38|
|     Ice Hockey|    2600|1980|        0.49|              11.38|
|        Freeway|    2600|1980|        0.34|              11.38|
|         Bridge|    2600|1980|        0.27|              11.38|
|       Checkers|    2600|1980|        0.24|              11.38|
|    Donkey Kong|    2600|1981|        1.46|              35.77|
+---------------+--------+----+------------+-------------------+
only showing top 10 rows



## 5. Продажи по годам, платформе и региону

Для выполнения данного пункта необходимо:

- преобразовать данные, объединив региональные продажи в один столбец;
- сгруппировать данные по `Year`, `Platform` и `Region`;
- рассчитать суммарные продажи.

Это позволит анализировать распределение продаж по регионам.

In [37]:
from pyspark.sql.functions import expr

df_unpivot = df_step_3.selectExpr(
    "Year",
    "Platform",
    "stack(4, \
        'NA', NA_Sales, \
        'EU', EU_Sales, \
        'JP', JP_Sales, \
        'Other', Other_Sales) as (Region, Sales)"
)

df_unpivot.show(5)

+----+--------+------+-----+
|Year|Platform|Region|Sales|
+----+--------+------+-----+
|2006|     Wii|    NA|41.49|
|2006|     Wii|    EU|29.02|
|2006|     Wii|    JP| 3.77|
|2006|     Wii| Other| 8.46|
|1985|     NES|    NA|29.08|
+----+--------+------+-----+
only showing top 5 rows



In [39]:
sales_by_year_platform_region = df_unpivot.groupBy(
    "Year", "Platform", "Region"
).agg(
    spark_sum("Sales").alias("Total_Sales_By_Year_Platform_Region")
)

In [40]:
sales_by_year_platform_region = sales_by_year_platform_region.withColumn(
    "Total_Sales_By_Year_Platform_Region",
    spark_round(col("Total_Sales_By_Year_Platform_Region"), 2)
)

sales_by_year_platform_region.orderBy("Year").show(10)

+----+--------+------+-----------------------------------+
|Year|Platform|Region|Total_Sales_By_Year_Platform_Region|
+----+--------+------+-----------------------------------+
|1980|    2600|    JP|                                0.0|
|1980|    2600|    NA|                              10.59|
|1980|    2600|    EU|                               0.67|
|1980|    2600| Other|                               0.12|
|1981|    2600|    JP|                                0.0|
|1981|    2600|    NA|                               33.4|
|1981|    2600| Other|                               0.32|
|1981|    2600|    EU|                               1.96|
|1982|    2600|    NA|                              26.92|
|1982|    2600| Other|                               0.31|
+----+--------+------+-----------------------------------+
only showing top 10 rows



In [41]:
df_step_4 = df_unpivot.join(
    sales_by_year_platform_region,
    on=["Year", "Platform", "Region"],
    how="left"
)

df_step_4.show(10)

+----+--------+------+-----+-----------------------------------+
|Year|Platform|Region|Sales|Total_Sales_By_Year_Platform_Region|
+----+--------+------+-----+-----------------------------------+
|2006|     Wii|    NA|41.49|                               71.3|
|2006|     Wii|    EU|29.02|                              43.84|
|2006|     Wii|    JP| 3.77|                               9.15|
|2006|     Wii| Other| 8.46|                              13.56|
|1985|     NES|    NA|29.08|                              33.31|
|1985|     NES|    EU| 3.58|                               4.68|
|1985|     NES|    JP| 6.81|                              14.54|
|1985|     NES| Other| 0.77|                               0.91|
|2008|     Wii|    NA|15.85|                              98.77|
|2008|     Wii|    EU|12.88|                              47.36|
+----+--------+------+-----+-----------------------------------+
only showing top 10 rows



In [42]:
df_step_4.select("Year", "Platform", "Region").distinct().count()

964

## 6. Доля продаж по годам, платформе и региону

Теперь необходимо рассчитать долю продаж каждой записи относительно общего объёма продаж:

- внутри одного года;
- для конкретной платформы;
- и региона.

Для этого используются оконные функции (Window Functions), которые позволяют выполнять вычисления внутри групп данных.

In [43]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("Year", "Platform", "Region")

In [47]:
df_step_5 = df_step_4.withColumn(
    "Share",
    when(
        spark_sum("Sales").over(window_spec) == 0,
        0
    ).otherwise(
        col("Sales") / spark_sum("Sales").over(window_spec)
    )
)

In [48]:
df_step_5 = df_step_5.withColumn(
    "Share",
    spark_round(col("Share"), 4)
)

df_step_5.show(10)

+----+--------+------+-----+-----------------------------------+------+
|Year|Platform|Region|Sales|Total_Sales_By_Year_Platform_Region| Share|
+----+--------+------+-----+-----------------------------------+------+
|1980|    2600|    EU| 0.26|                               0.67|0.3881|
|1980|    2600|    EU| 0.17|                               0.67|0.2537|
|1980|    2600|    EU| 0.07|                               0.67|0.1045|
|1980|    2600|    EU| 0.05|                               0.67|0.0746|
|1980|    2600|    EU| 0.04|                               0.67|0.0597|
|1980|    2600|    EU| 0.03|                               0.67|0.0448|
|1980|    2600|    EU| 0.02|                               0.67|0.0299|
|1980|    2600|    EU| 0.02|                               0.67|0.0299|
|1980|    2600|    EU| 0.01|                               0.67|0.0149|
|1980|    2600|    JP|  0.0|                                0.0|   0.0|
+----+--------+------+-----+-----------------------------------+

In [49]:
df_step_5.select(
    "Year",
    "Platform",
    "Region",
    "Sales",
    "Total_Sales_By_Year_Platform_Region",
    "Share"
).orderBy("Year", "Platform", "Region").show(20)

+----+--------+------+-----+-----------------------------------+------+
|Year|Platform|Region|Sales|Total_Sales_By_Year_Platform_Region| Share|
+----+--------+------+-----+-----------------------------------+------+
|1980|    2600|    EU| 0.26|                               0.67|0.3881|
|1980|    2600|    EU| 0.17|                               0.67|0.2537|
|1980|    2600|    EU| 0.07|                               0.67|0.1045|
|1980|    2600|    EU| 0.05|                               0.67|0.0746|
|1980|    2600|    EU| 0.04|                               0.67|0.0597|
|1980|    2600|    EU| 0.03|                               0.67|0.0448|
|1980|    2600|    EU| 0.02|                               0.67|0.0299|
|1980|    2600|    EU| 0.02|                               0.67|0.0299|
|1980|    2600|    EU| 0.01|                               0.67|0.0149|
|1980|    2600|    JP|  0.0|                                0.0|   0.0|
|1980|    2600|    JP|  0.0|                                0.0|

In [50]:
df_step_5.filter(col("Share").isNull()).count()

0

## 7. Доля продаж по издателю

На данном этапе рассчитаем долю продаж каждой игры относительно общего объёма продаж её издателя.

Для этого:
- сгруппируем данные по `Publisher`;
- рассчитаем общие продажи каждого издателя;
- добавим новый столбец `Publisher_Share`.

Этот показатель показывает, какую часть продаж конкретная игра занимает внутри всех продаж своего издателя.

In [51]:
publisher_window = Window.partitionBy("Publisher")

In [52]:
df_step_6 = df_step_3.withColumn(
    "Publisher_Total_Sales",
    spark_sum("Global_Sales").over(publisher_window)
)

In [53]:
df_step_6 = df_step_6.withColumn(
    "Publisher_Share",
    when(
        col("Publisher_Total_Sales") == 0,
        0
    ).otherwise(
        col("Global_Sales") / col("Publisher_Total_Sales")
    )
)

In [54]:
df_step_6 = df_step_6.withColumn(
    "Publisher_Total_Sales",
    spark_round(col("Publisher_Total_Sales"), 2)
).withColumn(
    "Publisher_Share",
    spark_round(col("Publisher_Share"), 4)
)

df_step_6.select(
    "Name",
    "Publisher",
    "Global_Sales",
    "Publisher_Total_Sales",
    "Publisher_Share"
).orderBy("Publisher", col("Publisher_Share").desc()).show(20)

+--------------------+--------------------+------------+---------------------+---------------+
|                Name|           Publisher|Global_Sales|Publisher_Total_Sales|Publisher_Share|
+--------------------+--------------------+------------+---------------------+---------------+
|      Panzer Tactics|     10TACLE Studios|        0.06|                 0.11|         0.5455|
|Boulder Dash: Rocks!|     10TACLE Studios|        0.03|                 0.11|         0.2727|
|Pirates: Legend o...|     10TACLE Studios|        0.02|                 0.11|         0.1818|
|Men of War: Assau...|          1C Company|        0.05|                  0.1|            0.5|
|      Off-Road Drive|          1C Company|        0.04|                  0.1|            0.4|
|King's Bounty: Ar...|          1C Company|        0.01|                  0.1|            0.1|
|               Alien|20th Century Fox ...|        0.79|                 1.94|         0.4072|
|    Fantastic Voyage|20th Century Fox ...|       

In [55]:
df_step_6.filter(col("Publisher_Share").isNull()).count()

0

## 8. Сумма продаж каждой игры

На данном этапе рассчитаем суммарные продажи каждой игры по всем регионам.

Для этого сложим:
- NA_Sales
- EU_Sales
- JP_Sales
- Other_Sales

Полученный результат добавим в новый столбец `Total_Game_Sales`.

Это позволит проверить корректность данных и использовать сумму продаж в дальнейшем анализе.

In [56]:
df_step_7 = df_step_6.withColumn(
    "Total_Game_Sales",
    col("NA_Sales") +
    col("EU_Sales") +
    col("JP_Sales") +
    col("Other_Sales")
)

In [57]:
df_step_7 = df_step_7.withColumn(
    "Total_Game_Sales",
    spark_round(col("Total_Game_Sales"), 2)
)

df_step_7.select(
    "Name",
    "NA_Sales",
    "EU_Sales",
    "JP_Sales",
    "Other_Sales",
    "Global_Sales",
    "Total_Game_Sales"
).show(10)

+--------------------+--------+--------+--------+-----------+------------+----------------+
|                Name|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|Total_Game_Sales|
+--------------------+--------+--------+--------+-----------+------------+----------------+
|          Wii Sports|   41.49|   29.02|    3.77|       8.46|       82.74|           82.74|
|   Super Mario Bros.|   29.08|    3.58|    6.81|       0.77|       40.24|           40.24|
|      Mario Kart Wii|   15.85|   12.88|    3.79|       3.31|       35.82|           35.83|
|   Wii Sports Resort|   15.75|   11.01|    3.28|       2.96|        33.0|            33.0|
|Pokemon Red/Pokem...|   11.27|    8.89|   10.22|        1.0|       31.37|           31.38|
|              Tetris|    23.2|    2.26|    4.22|       0.58|       30.26|           30.26|
|New Super Mario B...|   11.38|    9.23|     6.5|        2.9|       30.01|           30.01|
|            Wii Play|   14.03|     9.2|    2.93|       2.85|       29.02|      

In [58]:
df_step_7.withColumn(
    "Diff",
    spark_round(col("Global_Sales") - col("Total_Game_Sales"), 4)
).select(
    "Name",
    "Global_Sales",
    "Total_Game_Sales",
    "Diff"
).show(10)

+--------------------+------------+----------------+-----+
|                Name|Global_Sales|Total_Game_Sales| Diff|
+--------------------+------------+----------------+-----+
|          Wii Sports|       82.74|           82.74|  0.0|
|   Super Mario Bros.|       40.24|           40.24|  0.0|
|      Mario Kart Wii|       35.82|           35.83|-0.01|
|   Wii Sports Resort|        33.0|            33.0|  0.0|
|Pokemon Red/Pokem...|       31.37|           31.38|-0.01|
|              Tetris|       30.26|           30.26|  0.0|
|New Super Mario B...|       30.01|           30.01|  0.0|
|            Wii Play|       29.02|           29.01| 0.01|
|New Super Mario B...|       28.62|           28.61| 0.01|
|           Duck Hunt|       28.31|           28.31|  0.0|
+--------------------+------------+----------------+-----+
only showing top 10 rows



Проверка показала, что рассчитанный столбец `Total_Game_Sales` практически совпадает с `Global_Sales`.  
Небольшие расхождения в `0.01` связаны с округлением исходных данных.

## 9. Сохранение итогового датасета в Parquet

На заключительном этапе сохраним обработанный датасет в формат Parquet.

Parquet — это колоночный формат хранения данных, который хорошо подходит для аналитики и работы с большими наборами данных.

In [59]:
output_path = "/home/jovyan/work/data/processed/videogame_sales_parquet"

df_step_7.write.mode("overwrite").parquet(output_path)

In [60]:
df_parquet = spark.read.parquet(output_path)

df_parquet.show(5)

+----+-----+--------------------+--------+---------+---------------+--------+--------+--------+-----------+------------+-------------------+---------------------+---------------+----------------+
|Year| Rank|                Name|Platform|    Genre|      Publisher|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|Total_Sales_By_Year|Publisher_Total_Sales|Publisher_Share|Total_Game_Sales|
+----+-----+--------------------+--------+---------+---------------+--------+--------+--------+-----------+------------+-------------------+---------------------+---------------+----------------+
|2007|12351|      Panzer Tactics|      DS| Strategy|10TACLE Studios|    0.06|     0.0|     0.0|        0.0|        0.06|             611.13|                 0.11|         0.5455|            0.06|
|2007|14132|Boulder Dash: Rocks!|      DS|   Puzzle|10TACLE Studios|     0.0|    0.03|     0.0|        0.0|        0.03|             611.13|                 0.11|         0.2727|            0.03|
|2006|15709|Pirates:

In [61]:
df_parquet.printSchema()

root
 |-- Year: integer (nullable = true)
 |-- Rank: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Platform: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- NA_Sales: double (nullable = true)
 |-- EU_Sales: double (nullable = true)
 |-- JP_Sales: double (nullable = true)
 |-- Other_Sales: double (nullable = true)
 |-- Global_Sales: double (nullable = true)
 |-- Total_Sales_By_Year: double (nullable = true)
 |-- Publisher_Total_Sales: double (nullable = true)
 |-- Publisher_Share: double (nullable = true)
 |-- Total_Game_Sales: double (nullable = true)



In [62]:
print("Строк в итоговом датасете:", df_step_7.count())
print("Строк в parquet-файле:", df_parquet.count())

Строк в итоговом датасете: 16327
Строк в parquet-файле: 16327


## 10. Вывод

В ходе работы был обработан датасет Video Game Sales с использованием PySpark.

Были выполнены следующие этапы:
- чтение и нормализация данных;
- расчёт агрегированных показателей по годам;
- анализ продаж по платформам и регионам;
- вычисление долей продаж с использованием оконных функций;
- анализ долей продаж по издателям;
- проверка корректности данных;
- сохранение итогового датасета в формате Parquet.

Результаты показали корректность обработки данных, что подтверждается совпадением количества строк и проверкой вычисленных значений.